# nb_02 — Silver: conform, deduplicate, validate, convert

**Module 2.** One trustworthy row per workforce event.

1. **Deduplicate** replayed `event_id`s — keep the latest `ingest_ts`.
2. **Conform** the work-country code — `CORE_HR` emits ISO-3, `PAYROLL` emits ISO-2.
3. **Validate & quarantine** — negative pay, orphan employees, bad dates, unknown currency.
4. **Convert** pay amounts from local currency to **CAD** (the grid is in CAD).
5. Write `silver.workforce_event`.

> Note: many events (leaves, deployments) legitimately carry **no amount**. We
> quarantine *negative* pay, not null pay.

In [ ]:
from pyspark.sql import functions as F, Window as W
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
raw   = spark.table("bronze.workforce_events_raw")
wk    = spark.table("bronze.workers")
wkd   = spark.table("bronze.workers_delta")
fx    = spark.table("bronze.fx_rates")
print(f"bronze events (raw, incl replays): {raw.count():,}")

## 1. Deduplicate replays

The payroll system replays events; the same `event_id` arrives again with a later
`ingest_ts`. Keep the newest per `event_id`.

In [ ]:
w = W.partitionBy("event_id").orderBy(F.col("ingest_ts").desc())
dedup = raw.withColumn("_rn", F.row_number().over(w)).filter("_rn=1").drop("_rn")
print(f"after dedup: {dedup.count():,}  (removed {raw.count()-dedup.count():,})")

## 2. Conform the work-country code to ISO-3

`CORE_HR` sends ISO-3 (`CAN`), `PAYROLL` sends ISO-2 (`CA`). Map everything to
ISO-3 using a small static reference (the workforce spans a fixed set of
countries).

In [ ]:
iso = spark.createDataFrame([
    ("CA","CAN"),("US","USA"),("GB","GBR"),("FR","FRA"),("DE","DEU"),("JP","JPN"),
    ("AU","AUS"),("SG","SGP"),("BR","BRA"),("AE","ARE"),("KE","KEN"),("IN","IND"),
    ("MX","MEX")], ["_i2","_i3"])
conf = (dedup
    .withColumn("wcc3", F.when(F.length("work_country_code")==3, F.col("work_country_code")))
    .join(iso, dedup.work_country_code==iso._i2, "left")
    .withColumn("work_country_iso3", F.coalesce("wcc3","_i3"))
    .drop("_i2","_i3","wcc3"))

## 3. Validate & quarantine

| Rule | Reason |
|------|--------|
| `amount_local < 0` | `negative_pay` |
| `employee_id` doesn't resolve | `orphan_employee` |
| `event_date` before 2021-01-01 or future | `date_out_of_range` |
| unknown currency **or** unresolved country | `unresolved_reference` |

Null amounts are **valid** (non-pay events). Failing rows → quarantine.

In [ ]:
valid_emp = set(r.employee_id for r in
    wk.select("employee_id").union(wkd.select("employee_id")).distinct().collect())
valid_ccy = set(r.currency for r in fx.select("currency").distinct().collect())
flagged = (conf.withColumn("dq_reason",
    F.when(F.col("amount_local") < 0, "negative_pay")
     .when(~F.col("employee_id").isin(list(valid_emp)), "orphan_employee")
     .when((F.col("event_date") < F.lit("2021-01-01")) |
           (F.col("event_date") > F.current_date()), "date_out_of_range")
     .when(~F.col("local_currency").isin(list(valid_ccy)) |
           F.col("work_country_iso3").isNull(), "unresolved_reference")))
quarantine = flagged.filter("dq_reason IS NOT NULL")
clean = flagged.filter("dq_reason IS NULL").drop("dq_reason")
(quarantine.write.format("delta").mode("overwrite").option("overwriteSchema","true")
    .saveAsTable("silver.workforce_event_quarantine"))
quarantine.groupBy("dq_reason").count().orderBy("dq_reason").show()
print(f"clean rows continuing: {clean.count():,}")

## 4. Convert pay to CAD

The pay grid is in CAD, but international offices pay in local currency. Convert
`amount_local` → `amount_cad` on `(rate_month, currency)`. Non-pay events keep a
null amount.

In [ ]:
fx_lkp = fx.select(F.col("rate_month").alias("_rm"),
                   F.col("currency").alias("_ccy"), "cad_per_unit")
silver = (clean
    .withColumn("rate_month", F.date_format("event_date","yyyy-MM"))
    .join(fx_lkp, (F.col("rate_month")==F.col("_rm")) &
                  (F.col("local_currency")==F.col("_ccy")), "left")
    .withColumn("amount_cad",
        F.when(F.col("amount_local").isNull(), None)
         .otherwise(F.round(F.col("amount_local")*F.coalesce("cad_per_unit",F.lit(1.0)),2)))
    .select("event_id", F.to_date("event_date").alias("event_date"),
            "employee_id","cost_center_id",
            "classification_group", F.col("classification_level").cast("int").alias("classification_level"),
            "event_type",
            F.col("amount_local").cast("decimal(18,2)").alias("amount_local"),
            "local_currency", "work_country_iso3",
            F.col("amount_cad").cast("decimal(18,2)").alias("amount_cad"),
            "source_system", F.to_timestamp("ingest_ts").alias("ingest_ts")))
(silver.write.format("delta").mode("overwrite").option("overwriteSchema","true")
    .saveAsTable("silver.workforce_event"))
print(f"silver.workforce_event: {silver.count():,} rows")
silver.show(5, truncate=False)